In [ ]:
!wget https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv

--2025-09-14 11:43:06--  https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26129701 (25M) [text/plain]
Saving to: ‘combined_dataset.csv’

combined_dataset.cs 100%[===================>]  24.92M  --.-KB/s    in 0.1s    

2025-09-14 11:43:08 (198 MB/s) - ‘combined_dataset.csv’ saved [26129701/26129701]



In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    XLMRobertaModel,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from collections import Counter
import time
from datetime import datetime
import os
from dataclasses import dataclass
from typing import Optional, Dict, Any

In [ ]:
class MultiTaskRobertaModel(nn.Module):
    def __init__(self, model_name, num_ner_labels, num_pos_labels,
                 dropout_rate=0.1):
        super().__init__()

        self.roberta = XLMRobertaModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_rate)

        hidden_size = self.roberta.config.hidden_size

        self.ner_classifier = nn.Linear(hidden_size, num_ner_labels)
        self.pos_classifier = nn.Linear(hidden_size, num_pos_labels)

        self.num_ner_labels = num_ner_labels
        self.num_pos_labels = num_pos_labels

    def forward(self, input_ids=None, attention_mask=None,
                ner_labels=None, pos_labels=None, **kwargs):

        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)

        ner_logits = self.ner_classifier(sequence_output)
        pos_logits = self.pos_classifier(sequence_output)

        total_loss = None
        losses = {}

        if ner_labels is not None or pos_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            total_loss = 0

            if ner_labels is not None:
                ner_loss = loss_fct(ner_logits.view(-1, self.num_ner_labels), ner_labels.view(-1))
                losses['ner_loss'] = ner_loss
                total_loss += ner_loss

            if pos_labels is not None:
                pos_loss = loss_fct(pos_logits.view(-1, self.num_pos_labels), pos_labels.view(-1))
                losses['pos_loss'] = pos_loss
                total_loss += pos_loss

        return {
            'loss': total_loss,
            'ner_logits': ner_logits,
            'pos_logits': pos_logits,
            'hidden_states': outputs.hidden_states if hasattr(outputs, 'hidden_states') else None,
            'attentions': outputs.attentions if hasattr(outputs, 'attentions') else None,
            **losses
        }


class MultiTaskDataCollator:
    def __init__(self, tokenizer, padding=True, max_length=None, pad_to_multiple_of=None):
        self.tokenizer = tokenizer
        self.padding = padding
        self.max_length = max_length
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        if "ner_labels" in features[0]:
            sequence_length = batch["input_ids"].shape[1]
            padding_side = self.tokenizer.padding_side

            for label_name in ["ner_labels", "pos_labels"]:
                if label_name in features[0]:
                    if padding_side == "right":
                        batch[label_name] = [
                            f[label_name] + [-100] * (sequence_length - len(f[label_name]))
                            for f in features
                        ]
                    else:
                        batch[label_name] = [
                            [-100] * (sequence_length - len(f[label_name])) + f[label_name]
                            for f in features
                        ]

                    batch[label_name] = torch.tensor(batch[label_name], dtype=torch.long)

        return batch


class MultiTaskRoBERTa:
    def __init__(self, dataset, model_name="xlm-roberta-small"):
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dataset = dataset
        self.max_length = 64

        self.ner_label2id, self.ner_id2label = self.get_labels('NER_TAG')
        self.pos_label2id, self.pos_id2label = self.get_labels('POS_TAG')

        self.num_ner_labels = len(self.ner_label2id)
        self.num_pos_labels = len(self.pos_label2id)

        self.sentences = self.get_sentences()

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        self.model = MultiTaskRobertaModel(
            model_name=self.model_name,
            num_ner_labels=self.num_ner_labels,
            num_pos_labels=self.num_pos_labels,
        ).to(self.device)

        self.data_collator = MultiTaskDataCollator(
            tokenizer=self.tokenizer,
            padding=True
        )

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.trainer = None

        print(f"Multi-task RoBERTa initialized: {self.model_name}")

    def get_labels(self, column_name):
        tags = sorted(list(set(self.dataset[column_name].values)))
        label2id = {tag: i for i, tag in enumerate(tags)}
        id2label = {v: k for k, v in label2id.items()}
        return label2id, id2label

    def get_sentences(self):
        def to_tuples(group):
            return list(zip(
                group["WORD"].values,
                group["NER_TAG"].values,
                group["POS_TAG"].values,
            ))

        sentences = self.dataset.groupby("SENTENCE #").apply(
            to_tuples, include_groups=False
        ).tolist()
        return sentences

    def align_labels_with_tokens(self, words, ner_labels, pos_labels):
        tokenized_inputs = self.tokenizer(
            words,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            is_split_into_words=True,
            return_offsets_mapping=True
        )

        aligned_ner_labels = []
        aligned_pos_labels = []

        word_ids = tokenized_inputs.word_ids()
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_ner_labels.append(-100)
                aligned_pos_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_ner_labels.append(ner_labels[word_idx])
                aligned_pos_labels.append(pos_labels[word_idx])
            else:
                aligned_ner_labels.append(-100)
                aligned_pos_labels.append(-100)
            previous_word_idx = word_idx

        return {
            'input_ids': tokenized_inputs['input_ids'],
            'attention_mask': tokenized_inputs['attention_mask'],
            'ner_labels': aligned_ner_labels,
            'pos_labels': aligned_pos_labels,
        }

    def prepare_dataset(self, sentences):
        all_input_ids = []
        all_attention_masks = []
        all_ner_labels = []
        all_pos_labels = []
        skipped_sentences = 0

        for sentence in sentences:
            if len(sentence) == 0:
                skipped_sentences += 1
                continue

            words = [item[0] for item in sentence]
            ner_tags = [item[1] for item in sentence]
            pos_tags = [item[2] for item in sentence]

            ner_label_indices = [self.ner_label2id[tag] for tag in ner_tags]
            pos_label_indices = [self.pos_label2id[tag] for tag in pos_tags]

            aligned_data = self.align_labels_with_tokens(
                words, ner_label_indices, pos_label_indices
            )

            all_input_ids.append(aligned_data['input_ids'])
            all_attention_masks.append(aligned_data['attention_mask'])
            all_ner_labels.append(aligned_data['ner_labels'])
            all_pos_labels.append(aligned_data['pos_labels'])

        dataset = Dataset.from_dict({
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'ner_labels': all_ner_labels,
            'pos_labels': all_pos_labels,
        })

        return dataset

    def prepare_data_splits(self, test_size=0.1, val_size=0.1):
        sentences_temp, sentences_test = train_test_split(
            self.sentences, test_size=test_size, random_state=42
        )

        val_size_adjusted = val_size / (1 - test_size)
        sentences_train, sentences_val = train_test_split(
            sentences_temp, test_size=val_size_adjusted, random_state=42
        )

        self.train_dataset = self.prepare_dataset(sentences_train)
        self.val_dataset = self.prepare_dataset(sentences_val)
        self.test_dataset = self.prepare_dataset(sentences_test)

        print(f"Data splits prepared - Train: {len(sentences_train)}, Val: {len(sentences_val)}, Test: {len(sentences_test)}")

        return self.train_dataset, self.val_dataset, self.test_dataset

    def compute_metrics(self, eval_pred):
        predictions = eval_pred.predictions
        labels = eval_pred.label_ids

        ner_predictions = np.argmax(predictions[0], axis=2)
        pos_predictions = np.argmax(predictions[1], axis=2)

        ner_labels = labels['ner_labels'] if isinstance(labels, dict) else labels[0]
        pos_labels = labels['pos_labels'] if isinstance(labels, dict) else labels[1]

        def compute_task_metrics(predictions, labels, task_name):
            true_predictions = [
                [p for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]
            true_labels = [
                [l for (p, l) in zip(prediction, label) if l != -100]
                for prediction, label in zip(predictions, labels)
            ]

            flat_true_labels = [label for sublist in true_labels for label in sublist]
            flat_predictions = [pred for sublist in true_predictions for pred in sublist]

            if len(flat_true_labels) == 0:
                return {f"{task_name}_accuracy": 0.0, f"{task_name}_f1": 0.0,
                       f"{task_name}_precision": 0.0, f"{task_name}_recall": 0.0}

            accuracy = accuracy_score(flat_true_labels, flat_predictions)
            precision, recall, f1, _ = precision_recall_fscore_support(
                flat_true_labels, flat_predictions, average='weighted', zero_division=0
            )

            return {
                f"{task_name}_accuracy": float(accuracy),
                f"{task_name}_f1": float(f1),
                f"{task_name}_precision": float(precision),
                f"{task_name}_recall": float(recall),
            }

        ner_metrics = compute_task_metrics(ner_predictions, ner_labels, "ner")
        pos_metrics = compute_task_metrics(pos_predictions, pos_labels, "pos")

        all_metrics = {**ner_metrics, **pos_metrics}
        avg_f1 = (ner_metrics["ner_f1"] + pos_metrics["pos_f1"]) / 2
        all_metrics["avg_f1"] = float(avg_f1)

        return all_metrics

    def setup_trainer(self, output_dir='./multitask_roberta_results', num_epochs=4,
                     train_batch_size=8, eval_batch_size=16):
        if self.train_dataset is None:
            print("No training dataset found. Run prepare_data_splits() first.")
            return None

        os.environ["WANDB_DISABLED"] = "true"

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=train_batch_size,
            per_device_eval_batch_size=eval_batch_size,
            warmup_steps=1000,
            weight_decay=0.01,
            learning_rate=2e-5,
            logging_dir='./multitask_logs',
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=200,
            save_steps=400,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="avg_f1",
            greater_is_better=True,
            report_to=[],
            seed=42,
            fp16=torch.cuda.is_available(),
            dataloader_num_workers=2,
            remove_unused_columns=False,
            push_to_hub=False,
            optim="adamw_torch",
            gradient_accumulation_steps=2,
            max_grad_norm=1.0,
            lr_scheduler_type="linear",
        )

        self.trainer = MultiTaskTrainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            data_collator=self.data_collator,
            processing_class=self.tokenizer,
            compute_metrics=self.compute_metrics,
        )

        print(f"Multi-task Trainer setup complete")
        return self.trainer

    def clear_gpu_memory(self):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    def train(self, save_model_path='./best_multitask_model'):
        if self.trainer is None:
            print("Trainer not setup. Run setup_trainer() first.")
            return None

        self.clear_gpu_memory()

        print("Starting training...")
        start_time = time.time()

        try:
            train_result = self.trainer.train()
            end_time = time.time()
            training_time = end_time - start_time

            print(f"Training completed in {training_time/60:.1f} minutes")
            print(f"Final train loss: {train_result.training_loss:.4f}")

            self.clear_gpu_memory()

            torch.save({
                'model_state_dict': self.model.state_dict(),
                'ner_label2id': self.ner_label2id,
                'pos_label2id': self.pos_label2id,
                'ner_id2label': self.ner_id2label,
                'pos_id2label': self.pos_id2label,
                'model_name': self.model_name
            }, f"{save_model_path}/multitask_model.pt")

            self.tokenizer.save_pretrained(save_model_path)
            print(f"Model saved to '{save_model_path}'")

            return train_result

        except Exception as e:
            print(f"Training failed: {e}")
            self.clear_gpu_memory()
            return None

    def evaluate(self, dataset=None):
        if self.trainer is None:
            print("Trainer not available.")
            return None

        if dataset is None:
            dataset = self.test_dataset

        if dataset is None:
            print("No dataset provided and no test dataset available.")
            return None

        self.clear_gpu_memory()

        try:
            eval_result = self.trainer.evaluate(dataset)

            print("Evaluation Results:")
            print(f"Loss: {eval_result['eval_loss']:.4f}")
            print(f"Average F1: {eval_result['eval_avg_f1']:.4f}")
            print(f"NER - Accuracy: {eval_result['eval_ner_accuracy']:.4f}, F1: {eval_result['eval_ner_f1']:.4f}")
            print(f"POS - Accuracy: {eval_result['eval_pos_accuracy']:.4f}, F1: {eval_result['eval_pos_f1']:.4f}")

            return eval_result

        except Exception as e:
            print(f"Evaluation failed: {e}")
            self.clear_gpu_memory()
            return None

    def predict(self, text):
        self.model.eval()

        if isinstance(text, str):
            words = text.split()
        else:
            words = text

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        encoding = {k: v.to(self.device) for k, v in encoding.items() if k != 'offset_mapping'}

        with torch.no_grad():
            outputs = self.model(**encoding)
            ner_predictions = torch.argmax(outputs['ner_logits'], dim=2)
            pos_predictions = torch.argmax(outputs['pos_logits'], dim=2)

        word_ids = self.tokenizer(words, is_split_into_words=True).word_ids()

        predicted_ner_tags = []
        predicted_pos_tags = []
        seen_word_ids = set()

        for i, word_id in enumerate(word_ids):
            if word_id is not None and word_id not in seen_word_ids and i < len(ner_predictions[0]):
                ner_tag_id = ner_predictions[0][i].item()
                pos_tag_id = pos_predictions[0][i].item()

                ner_tag = self.ner_id2label.get(ner_tag_id, 'O')
                pos_tag = self.pos_id2label.get(pos_tag_id, 'UNK')

                predicted_ner_tags.append(ner_tag)
                predicted_pos_tags.append(pos_tag)
                seen_word_ids.add(word_id)

        result = []
        for i, word in enumerate(words):
            if i < len(predicted_ner_tags):
                result.append((
                    word,
                    predicted_ner_tags[i],
                    predicted_pos_tags[i],
                ))
            else:
                result.append((word, 'O', 'UNK'))

        return result

    def detailed_predict(self, text):
        print(f"Analysis: '{text}'")
        predictions = self.predict(text)

        print(f"{'Word':<15} {'NER':<12} {'POS':<12}")
        for word, ner_tag, pos_tag in predictions:
            print(f"{word:<15} {ner_tag:<12} {pos_tag:<12}")

        entities = self.extract_entities_from_predictions(predictions)
        if entities:
            print(f"Entities: {', '.join(entities)}")

        return {
            'predictions': predictions,
            'entities': entities
        }

    def extract_entities_from_predictions(self, predictions):
        entities = []
        current_entity = []
        current_entity_type = None

        for word, ner_tag, pos_tag in predictions:
            if ner_tag.startswith('B-'):
                if current_entity:
                    entities.append(' '.join(current_entity))
                current_entity = [word]
                current_entity_type = ner_tag.split('-')[1]
            elif ner_tag.startswith('I-') and current_entity_type == ner_tag.split('-')[1]:
                current_entity.append(word)
            else:
                if current_entity:
                    entities.append(' '.join(current_entity))
                current_entity = []
                current_entity_type = None

        if current_entity:
            entities.append(' '.join(current_entity))

        return entities


class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        outputs = model(**inputs)
        loss = outputs.get('loss')
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.get('loss')

            ner_logits = outputs.get('ner_logits')
            pos_logits = outputs.get('pos_logits')

            ner_labels = inputs.get('ner_labels')
            pos_labels = inputs.get('pos_labels')

        if prediction_loss_only:
            return (loss, None, None)

        predictions = (ner_logits, pos_logits)
        labels = {
            'ner_labels': ner_labels,
            'pos_labels': pos_labels,
        }

        return (loss, predictions, labels)


def load_multitask_model(model_path, tokenizer_path=None):
    if tokenizer_path is None:
        tokenizer_path = model_path

    checkpoint = torch.load(f"{model_path}/multitask_model.pt")

    model = MultiTaskRobertaModel(
        model_name=checkpoint['model_name'],
        num_ner_labels=len(checkpoint['ner_label2id']),
        num_pos_labels=len(checkpoint['pos_label2id']),
    )

    model.load_state_dict(checkpoint['model_state_dict'])
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

    return model, tokenizer, checkpoint


def create_multitask_inference_pipeline(model_path):
    model, tokenizer, checkpoint = load_multitask_model(model_path)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    ner_id2label = checkpoint['ner_id2label']
    pos_id2label = checkpoint['pos_id2label']

    def predict(text, max_length=128):
        if isinstance(text, str):
            words = text.split()
        else:
            words = text

        encoding = tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )

        encoding = {k: v.to(device) for k, v in encoding.items() if k != 'offset_mapping'}

        with torch.no_grad():
            outputs = model(**encoding)
            ner_predictions = torch.argmax(outputs['ner_logits'], dim=2)
            pos_predictions = torch.argmax(outputs['pos_logits'], dim=2)

        word_ids = tokenizer(words, is_split_into_words=True).word_ids()

        predicted_ner_tags = []
        predicted_pos_tags = []
        seen_word_ids = set()

        for i, word_id in enumerate(word_ids):
            if word_id is not None and word_id not in seen_word_ids and i < len(ner_predictions[0]):
                ner_tag_id = ner_predictions[0][i].item()
                pos_tag_id = pos_predictions[0][i].item()

                ner_tag = ner_id2label.get(str(ner_tag_id), 'O')
                pos_tag = pos_id2label.get(str(pos_tag_id), 'UNK')

                predicted_ner_tags.append(ner_tag)
                predicted_pos_tags.append(pos_tag)
                seen_word_ids.add(word_id)

        result = []
        for i, word in enumerate(words):
            if i < len(predicted_ner_tags):
                result.append((
                    word,
                    predicted_ner_tags[i],
                    predicted_pos_tags[i],
                ))
            else:
                result.append((word, 'O', 'UNK'))

        return result

    return predict

In [ ]:
# Load your dataset with three annotation columns
dataset = pd.read_csv('combined_dataset.csv')  # Columns: SENTENCE #, WORD, NER_TAG, POS_TAG, LEMMA

In [ ]:
# Initialize multi-task model
mt_model = MultiTaskRoBERTa(dataset, model_name="xlm-roberta-base")

Multi-task RoBERTa initialized: xlm-roberta-base


In [ ]:
# Prepare data
train_ds, val_ds, test_ds = mt_model.prepare_data_splits()

Data splits prepared - Train: 31380, Val: 3923, Test: 3923


In [ ]:
trainer = mt_model.setup_trainer(
    output_dir='./multitask_results',
    num_epochs=4,
    train_batch_size=12,
)

Multi-task Trainer setup complete


In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore", message=".*XLMRobertaTokenizerFast.*")
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)
# Train
results = mt_model.train('./best_multitask_model')

Starting training...


Step,Training Loss,Validation Loss,Ner Accuracy,Ner F1,Ner Precision,Ner Recall,Pos Accuracy,Pos F1,Pos Precision,Pos Recall,Avg F1
200,2.842300,1.763778,0.910739,0.868707,0.834847,0.910739,0.659288,0.555926,0.580445,0.659288,0.712316
400,0.623500,0.396382,0.964838,0.956723,0.950567,0.964838,0.950545,0.949416,0.949206,0.950545,0.953070
600,0.326600,0.254202,0.974391,0.970111,0.967073,0.974391,0.964359,0.963726,0.963125,0.964359,0.966919
800,0.257100,0.220626,0.978004,0.977485,0.977810,0.978004,0.966230,0.965714,0.965481,0.966230,0.971599
1000,0.224700,0.188971,0.980564,0.980642,0.981443,0.980564,0.969609,0.969476,0.969533,0.969609,0.975059
1200,0.194700,0.163808,0.984082,0.983368,0.982879,0.984082,0.971203,0.971050,0.970987,0.971203,0.977209
1400,0.169600,0.155855,0.984624,0.984272,0.984263,0.984624,0.972435,0.972300,0.972243,0.972435,0.978286
1600,0.164600,0.147996,0.984337,0.984213,0.984515,0.984337,0.973551,0.973406,0.973313,0.973551,0.978809
1800,0.150300,0.141350,0.985325,0.984995,0.984955,0.985325,0.974476,0.974340,0.974267,0.974476,0.979667
2000,0.158300,0.134851,0.985229,0.985078,0.985258,0.985229,0.974964,0.974862,0.974971,0.974964,0.979970


Training completed in 12.5 minutes
Final train loss: 0.3408
Training failed: Parent directory ./best_multitask_model does not exist.


In [ ]:
# Evaluate all tasks
eval_results = mt_model.evaluate()


Evaluation Results:
Loss: 0.1171
Average F1: 0.9829
NER - Accuracy: 0.9862, F1: 0.9864
POS - Accuracy: 0.9794, F1: 0.9794


In [ ]:
# Predict all three tasks at once
predictions = mt_model.detailed_predict("Shqipëria do të marrë pjesë në Samitin e BE-së në Bruksel të hënën në 20 Maj 2020.")

Analysis: 'Shqipëria do të marrë pjesë në Samitin e BE-së në Bruksel të hënën në 20 Maj 2020.'
Word            NER          POS         
Shqipëria       B-VEND_1     PROPN       
do              O            PART        
të              O            PART        
marrë           O            VERB        
pjesë           O            NOUN        
në              O            ADP         
Samitin         B-EVENT      PROPN       
e               I-EVENT      DET         
BE-së           B-ORG        PROPN       
në              O            ADP         
Bruksel         B-VEND_0     PROPN       
të              B-DATE_1     DET         
hënën           I-DATE_1     NOUN        
në              O            ADP         
20              B-DATE_0     NUM         
Maj             I-DATE_0     NOUN        
2020.           I-DATE_0     NUM         
Entities: Shqipëria, Samitin e, BE-së, Bruksel, të hënën, 20 Maj 2020.
